In [ ]:
import qiskit
from qiskit_algorithms import QAOA
from qiskit_optimization import QuadraticProgram

In [ ]:
# W = [
#     [0, 10, 1, 3],
#     [10, 0, 4, 2],
#     [1, 4, 0, 2],
#     [3, 2, 2, 0]
# ]

W = [
    [0, 10, 1],
    [10, 0, 3],
    [1, 3, 0],
]


# W = [
#     [0, 2],
#     [2, 0]
# ]

# W = [
#     [  0.00,  26.57,  12.07,  47.19, 126.81,  35.69,  77.42,  76.09, 125.47,  78.93],
#     [ 40.15,   0.00,  41.21,  78.54, 118.78,  41.20,  91.83,  85.67, 134.36, 140.95],
#     [ 12.07,  70.52,   0.00,  39.15,  64.20,  92.26, 115.77, 117.75, 141.45,  39.78],
#     [ 62.83,  40.30,  39.15,   0.00,  48.47,  15.35,  96.51,  49.51,  90.94,   0.77],
#     [172.35, 134.38,  64.20,  27.94,   0.00,  48.81,  26.91,  23.45,  31.52,  28.75],
#     [ 90.54,  41.20,  42.21,   3.04,  49.72,   0.00,  68.40,  48.17, 117.34,   8.62],
#     [ 76.64,  89.36,  87.14,  52.94,  26.91,  94.27,   0.00,  12.55,   8.96,  67.32],
#     [ 77.27,  85.52,  87.38,  52.82,  23.46,  48.17,   6.24,   0.00,   9.16,  67.09],
#     [130.99, 133.05, 110.34,  73.83,  31.52,  83.61,  11.19,   8.96,   0.00,  58.48],
#     [113.36,  74.74,  40.17,   0.77,  38.26,   8.56, 134.50,  65.68,  58.58,   0.00],
# ]

# W = W[:4][:4]

V = [50, 5, 10]

L = 50


n = len(W)

print(f"Number of cities : {n}")
print(f"Number of variables : {n**2}")
print(f"Size of hamiltonian : {2**(n**2)}")



Number of cities : 3
Number of variables : 9
Size of hamiltonian : 512


In [ ]:
qp = QuadraticProgram(name="TSP-City-Selection")

# --- Variables: only x[i][p] ---
for i in range(n):
    for p in range(n):
        qp.binary_var(name=f"x_{i}_{p}")

def x(i, p): return f"x_{i}_{p}"

# --- Penalty scalar ---
P = max(V)/L
print(P)
# "each unit of fuel is worth max_value/L in penalty"
# routes exceeding L become naturally unattractive

# --- Objective: maximize value - penalty for excess fuel ---
# maximize sum_i V[i] * sum_p x[i][p]
# minus P * max(0, fuel - L)
# but max(0, ...) is non-linear, so instead penalize all fuel directly:
# maximize sum V[i]*x[i][p] - P * sum W[i][j]*x[i][p]*x[j][p+1]
# which naturally discourages high fuel routes

linear_terms = {}
for i in range(n):
    for p in range(n):
        linear_terms[x(i, p)] = V[i]

quadratic_terms = {}
for i in range(n):
    for j in range(n):
        if i != j:
            for p in range(n - 1):
                key = (x(i, p), x(j, p + 1))
                quadratic_terms[key] = - P * W[i][j] # quadratic_terms.get(key, 0)

qp.maximize(linear=linear_terms, quadratic=quadratic_terms)

# --- Constraint 1: each city visited AT MOST once ---
for i in range(n):
    qp.linear_constraint(
        linear={x(i, p): 1 for p in range(n)},
        sense="<=",
        rhs=1,
        name=f"city_{i}_at_most_once"
    )

# --- Constraint 2: each position has AT MOST one city ---
for p in range(n):
    qp.linear_constraint(
        linear={x(i, p): 1 for i in range(n)},
        sense="<=",
        rhs=1,
        name=f"position_{p}_at_most_one"
    )

# --- Constraint 3: positions must be contiguous ---
for p in range(1, n):
    qp.linear_constraint(
        linear={**{x(i, p): 1 for i in range(n)},
                **{x(i, p-1): -1 for i in range(n)}},
        sense="<=",
        rhs=0,
        name=f"contiguous_positions_{p}"
    )

print(qp.export_as_lp_string())

0
\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: TSP-City-Selection

Maximize
 obj: 50 x_0_0 + 50 x_0_1 + 50 x_0_2 + 5 x_1_0 + 5 x_1_1 + 5 x_1_2 + 10 x_2_0
      + 10 x_2_1 + 10 x_2_2
Subject To
 city_0_at_most_once: x_0_0 + x_0_1 + x_0_2 <= 1
 city_1_at_most_once: x_1_0 + x_1_1 + x_1_2 <= 1
 city_2_at_most_once: x_2_0 + x_2_1 + x_2_2 <= 1
 position_0_at_most_one: x_0_0 + x_1_0 + x_2_0 <= 1
 position_1_at_most_one: x_0_1 + x_1_1 + x_2_1 <= 1
 position_2_at_most_one: x_0_2 + x_1_2 + x_2_2 <= 1
 contiguous_positions_1: - x_0_0 + x_0_1 - x_1_0 + x_1_1 - x_2_0 + x_2_1 <= 0
 contiguous_positions_2: - x_0_1 + x_0_2 - x_1_1 + x_1_2 - x_2_1 + x_2_2 <= 0

Bounds
 0 <= x_0_0 <= 1
 0 <= x_0_1 <= 1
 0 <= x_0_2 <= 1
 0 <= x_1_0 <= 1
 0 <= x_1_1 <= 1
 0 <= x_1_2 <= 1
 0 <= x_2_0 <= 1
 0 <= x_2_1 <= 1
 0 <= x_2_2 <= 1

Binaries
 x_0_0 x_0_1 x_0_2 x_1_0 x_1_1 x_1_2 x_2_0 x_2_1 x_2_2
End



In [ ]:
from qiskit_optimization.converters import (
    QuadraticProgramToQubo,   # folds constraints into objective via penalties
    LinearEqualityToPenalty,  # sub-step: converts == constraints to penalty terms
    MinimizeToMaximize,       # flips sign if needed
    IntegerToBinary,          # converts integer vars to binary (not needed here)
)
from qiskit_optimization.translators import to_ising
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
import numpy as np

# Step 1: QP → QUBO  
qubo_converter = QuadraticProgramToQubo()
qubo = qubo_converter.convert(qp)
print(qubo)

# Step 2: QUBO → Ising Hamiltonian 
ising_operator, offset = to_ising(qubo)


minimize 3724*contiguous_positions_1@int_slack@0^2 + 14896*contiguous_positions_1@int_slack@0*contiguous_positions_1@int_slack@1 + 14896*contiguous_positions_1@int_slack@1^2 + 3724*contiguous_positions_2@int_slack@0^2 + 14896*contiguous_positions_2@int_slack@0*contiguous_positions_2@int_slack@1 + 14896*contiguous_positions_2@int_slack@1^2 - 7448*x_0_0*contiguous_positions_1@int_slack@0 - 14896*x_0_0*contiguous_positions_1@int_slack@1 + 3724*x_0_0^2 - 7252*x_0_0*x_0_1 + 196*x_0_0*x_0_2 + 7644*x_0_0*x_1_0 - 7448*x_0_0*x_1_1 + 7644*x_0_0*x_2_0 - 7448*x_0_0*x_2_1 + 7448*x_0_1*contiguous_positions_1@int_slack@0 + 14896*x_0_1*contiguous_positions_1@int_slack@1 - 7448*x_0_1*contiguous_positions_2@int_slack@0 - 14896*x_0_1*contiguous_positions_2@int_slack@1 + 7448*x_0_1^2 - 7252*x_0_1*x_0_2 - 7448*x_0_1*x_1_0 + 15092*x_0_1*x_1_1 - 7448*x_0_1*x_1_2 - 7448*x_0_1*x_2_0 + 15092*x_0_1*x_2_1 - 7448*x_0_1*x_2_2 + 7448*x_0_2*contiguous_positions_2@int_slack@0 + 14896*x_0_2*contiguous_positions_2@int_s

In [ ]:
from qiskit_aer.primitives import Sampler as AerSampler

sampler = AerSampler(
    backend_options={
        "method": "statevector", #matrix_product_state
        "max_parallel_threads": 0,      # 0 = all cores
        "max_parallel_experiments": 0,
        "max_parallel_shots": 0
    },
    run_options={
        "shots": 1024
    }
)

optimizer = COBYLA()

In [ ]:

# Step 3: feed to QAOA
qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=5,               # p layers — more layers = better approximation, more circuit depth
    initial_point=None,   # optional initial [β, γ] parameters, length = 2p
    mixer=None,           # default X mixer; can provide custom circuit for constrained problems
)
result = qaoa.compute_minimum_eigenvalue(ising_operator)
print(result)


# Step 4: interpret result back in terms of original variables
bitstring = result.best_measurement["bitstring"]           # e.g. "1000010000100001"
x_array   = np.array([int(b) for b in bitstring])         # → [1,0,0,0,0,1,0,0,...]

decoded = qubo_converter.interpret(x_array)
print(decoded)

{   'aux_operators_evaluated': None,
    'best_measurement': {   'bitstring': '0000010001100',
                            'probability': 0.0029296875,
                            'state': 140,
                            'value': np.complex128(-38089.5+0j)},
    'cost_function_evals': np.int64(80),
    'eigenstate': {2735: 0.0009765625, 497: 0.0009765625, 3301: 0.0009765625, 3848: 0.0009765625, 6763: 0.0009765625, 7667: 0.0009765625, 1664: 0.0009765625, 16: 0.0009765625, 3132: 0.0009765625, 5947: 0.0009765625, 8102: 0.0009765625, 7088: 0.0009765625, 3662: 0.0009765625, 1201: 0.0009765625, 2630: 0.0009765625, 795: 0.0009765625, 7970: 0.0009765625, 2760: 0.0009765625, 5883: 0.0009765625, 419: 0.0009765625, 2584: 0.0009765625, 4203: 0.0009765625, 5985: 0.0009765625, 1490: 0.0009765625, 1872: 0.0009765625, 3083: 0.0009765625, 6804: 0.0009765625, 8076: 0.0009765625, 5097: 0.0009765625, 3039: 0.0009765625, 4934: 0.0009765625, 7872: 0.0009765625, 2709: 0.0009765625, 5569: 0.0009765625, 2906:

In [ ]:
import numpy as np

def from_bstring_to_circuit(bstring) :
    bstring = bstring[::-1]
    buf = [n + 1 for i in range(n)]
    for i in range(n) :
        for j in range(n) :
            if bstring[i*n + j] == 1 :
                buf[i] = j

    order = np.argsort(buf)
    out_str = f""
    for i in range(n) :
        if i == (n - 1) or buf[order[i + 1]] > n :
            out_str += f" {order[i]}"
            break
        out_str += f" {order[i]} ->"
    print(out_str)
    return order

from_bstring_to_circuit(decoded)

 1


array([1, 0, 2])